# 1. Laden → Filtern → VCG-Transformation → Annotation → Visualisierung

Dieses Notebook demonstriert den kompletten Vorverarbeitungs- und Annotations-Teil von `vcgsuite` an einer echten EASI-Aufnahme — und dient gleichzeitig als **Funktionskontrolle**: wenn alle Zellen fehlerfrei durchlaufen und die Plots sinnvoll aussehen (klare QRS-Komplexe, plausible Marker-Positionen), arbeitet die Bibliothek wie vorgesehen.

Ablauf:
1. Rohdaten laden, filtern, in ein Vektorkardiogramm (VCG) transformieren
2. Frenet-Serret-Kinematik berechnen (Krümmung, Torsion, Geschwindigkeit, ...)
3. R-Peaks und R-Turns detektieren
4. Alle Beats hierarchisch annotieren (12 Landmarken pro Herzschlag)
5. Ergebnis visualisieren (2D-Zeitreihe + 3D-VCG-Trajektorie mit Markern)

Siehe auch: [`docs/vcg_analysis_pipeline.md`](../../docs/vcg_analysis_pipeline.md) und [`docs/vcg_beat_annotation.md`](../../docs/vcg_beat_annotation.md).

In [1]:
import vcgsuite as ecg
from vcgsuite.viz.r_peaks import plot_r_peak_detection
from vcgsuite.viz.vcg_annotated import build_plot_data, select_time_window, plot_signal_segments_and_markers, plot_vcg_3d

print(f"vcgsuite {ecg.__version__}")

vcgsuite 0.1.0


## Konfiguration

`DATA_PATH` zeigt auf die lokale Beispielaufnahme unter `sample_data/` (liegt bewusst außerhalb des Repos, siehe README — personenbezogene Rohdaten werden nicht versioniert). Passe den Pfad an, falls deine Daten woanders liegen.

Es wird nur ein `DURATION_S`-Ausschnitt verarbeitet, damit das Notebook schnell durchläuft — für eine echte Analyse einfach `duration=None` setzen (verarbeitet die gesamte Aufnahme).

In [2]:
DATA_PATH = "../../../sample_data/-I-2025-4-6_6min_2brust_2bauchOhne_2bauchmit.txt"
MODE = "easi"          # oder "12ch"
START_SEC = 0.0
DURATION_S = 90.0      # Ausschnitt für eine schnelle Demo; None = ganze Aufnahme

## 1. Laden → Filtern → VCG-Transformation

`ecg.load_and_process()` übernimmt in einem Aufruf: Rohdatei laden + Zeitfenster schneiden → ZapLine (Netzfrequenz) + FIR-Bandpass → EASI/12-Kanal → Frank-XYZ-Transformation.

In [3]:
df_analysis = ecg.load_and_process(
    DATA_PATH,
    mode=MODE,
    start_sec=START_SEC,
    duration=DURATION_S,
)

print(df_analysis.attrs)
df_analysis.head()


  Pipeline : EASI  |  -I-2025-4-6_6min_2brust_2bauchOhne_2bauchmit.txt
  Proband  : 2025  →  S2025
EASI  |  Proband: 2025  |  Fenster: 0.0 s  →  90.0 s  (22500 Samples = 90.0 s)
       Spalten: ['R', 'M', 'L']
  [Filter] ZapLine 50.0 Hz  →  FIR HP 1.5 Hz / LP 37.5 Hz
Power of components removed by DSS: 0.08
ZapLine: 50.0 Hz, bis 2. Harmonische, nremove=1
FIR: HP 1.5 Hz (201 Taps)  →  LP 37.5 Hz (21 Taps)
  [VCG]    EASI → Frank XYZ  (W · T)

  df_analysis: (22500, 7)  |  Proband: S2025

{'proband_id': '2025', 'subject_id': 'S2025', 'mode': 'easi', 'fs': 250}


,Time,X,Y,Z,V_IS,V_ES,V_AS
0,0.000,0.000640,0.000454,0.001267,0.000113,-0.001832,0.002055
1,0.004,0.035869,0.000921,0.235187,-0.034469,-0.394367,0.184186
2,0.008,0.050779,-0.016448,0.414529,-0.075034,-0.709209,0.289097
3,0.012,0.035050,-0.058128,0.503030,-0.121102,-0.890326,0.275375
4,0.016,-0.007182,-0.115263,0.493371,-0.164222,-0.918134,0.153637


## 2. Frenet-Serret-Kinematik

Ergänzt Geschwindigkeit, Beschleunigung, Jerk, Krümmung κ, Torsion τ sowie sphärische Koordinaten (r, θ, φ) — die Grundlage für die orientierungsunabhängige Beat-Annotation (siehe `docs/vcg_beat_annotation.md`).

In [4]:
df_analysis, dt = ecg.compute_vcg_kinematics(df_analysis)
df_analysis.attrs["fs"] = df_analysis.attrs.get("fs", 1.0 / dt)

df_analysis[["Time", "X", "Y", "Z", "V_abs", "A_abs", "Curvature", "Torsion", "r", "theta", "phi"]].describe()

,Time,X,Y,Z,V_abs,A_abs,Curvature,Torsion,r,theta,phi
count,22500.000000,22500.000000,22500.000000,22500.000000,22500.000000,22500.000000,22500.000000,22500.000000,22500.000000,22500.000000,22500.000000
mean,44.998000,-0.000068,-0.000071,0.000039,4.529222,465.425159,87.893640,-3.439778,0.084744,-0.873403,1.697829
std,25.981339,0.067125,0.055007,0.101584,7.846410,864.216151,429.130745,219.895663,0.103290,1.795835,0.626326
min,0.000000,-0.138724,-0.207155,-0.252074,0.063742,6.928931,0.187145,-9231.141632,0.000583,-3.141389,0.023476
25%,22.499000,-0.043651,-0.032637,-0.041327,1.384428,131.949725,9.804243,-17.782043,0.027343,-2.452144,1.180375
50%,44.998000,-0.009281,-0.009200,-0.008684,2.281253,215.846517,23.843556,-0.883427,0.061311,-1.772475,1.845987
75%,67.497000,0.010282,0.018889,0.008452,3.783433,350.136279,70.871622,10.698698,0.096794,0.723224,2.146051
max,89.996000,0.456511,0.343593,0.691118,61.030606,7203.115218,35256.428187,22554.887124,0.737068,3.141504,3.120459


## 3. R-Peak- und R-Turn-Detektion

In [5]:
r_peak_times, r_peak_values = ecg.detect_r_peaks(df_analysis)
r_turn_times, r_turn_values = ecg.detect_r_turn(df_analysis, r_peak_times)

duration = df_analysis["Time"].iloc[-1] - df_analysis["Time"].iloc[0]
bpm = len(r_peak_times) / (duration / 60.0)
print(f"{len(r_peak_times)} R-Peaks erkannt  |  Ø {bpm:.1f} bpm")

Sampling-Rate:     250.0 Hz
Fensterbreite:     10.0 s  (2500 Samples)
Detektierte Peaks: 101
Ø RR-Abstand:      893.1 ms
Ø HF:              67.2 bpm
RR-Bereich:        680 – 1128 ms
R_turn gefunden:   101 / 101
Ø Δ R_peak→R_turn: 12.0 ms  (erwartet ~12 ms)
Min/Max Δ:         12.0 / 12.0 ms
101 R-Peaks erkannt  |  Ø 67.3 bpm


In [6]:
fig = plot_r_peak_detection(df_analysis, r_peak_times, r_peak_values)
fig.show()

**Kontrolle:** Die rote Markerlinie sollte exakt auf den Gipfeln von `A_abs` sitzen, ein Marker pro Herzschlag, keine Doppel- oder Fehldetektionen bei ruhigem Signal.

## 4. Hierarchische Beat-Annotation

Annotiert alle 12 Landmarken (P/Q/R/S/T-Komplex) pro Herzschlag über Multi-Feature-Konsens-Voting auf der Kinematik (siehe `docs/vcg_beat_annotation.md`).

In [7]:
df_annotations = ecg.annotate_all_beats(df_analysis, r_peak_times, r_turn_times)
df_annotations.head()

Annotierte Beats: 101

Marker        gefunden   Ø rel. Zeit
----------------------------------------
  R_turn            101       +12.0 ms
  S_on              101       +29.1 ms
  Q_on              100       -60.8 ms
  Q_off             101       -20.0 ms
  P_peak            100      -133.0 ms
  P_on              100      -182.7 ms
  P_off             100       -62.1 ms
  S_off             101       +62.1 ms
  T_on              101      +158.2 ms
  T_turn1           101      +266.0 ms
  T_turn2           101      +362.7 ms
  T_off             101      +458.2 ms


,beat_id,t_R_peak+,t_R_turn,t_S_on,t_Q_on,t_Q_off,t_P_peak,t_P_on,t_P_off,t_S_off,t_T_on,t_T_turn1,t_T_turn2,t_T_off
0,0,0.012,0.024,0.032,NaN,-0.008,NaN,NaN,NaN,0.104,0.184,0.264,0.360,0.434
1,1,0.772,0.784,0.800,0.716,0.752,0.632,0.556,0.724,0.864,0.942,1.024,1.108,1.234
2,2,1.584,1.596,1.612,1.540,1.564,1.452,1.402,1.542,1.662,1.786,1.868,1.956,2.088
3,3,2.460,2.472,2.484,2.396,2.440,2.332,2.286,2.404,2.512,2.614,2.720,2.832,2.952
4,4,3.344,3.356,3.376,3.282,3.324,3.216,3.174,3.266,3.426,3.530,3.612,3.692,3.794


In [8]:
# Kontrolle: Trefferquote pro Marker (sollte für die meisten Marker deutlich > 80% liegen)
marker_cols = [c for c in df_annotations.columns if c.startswith("t_")]
coverage = (df_annotations[marker_cols].notna().mean() * 100).round(1).sort_values()
coverage

t_Q_on        99.0
t_P_peak      99.0
t_P_on        99.0
t_P_off       99.0
t_R_peak+    100.0
t_R_turn     100.0
t_S_on       100.0
t_Q_off      100.0
t_S_off      100.0
t_T_on       100.0
t_T_turn1    100.0
t_T_turn2    100.0
t_T_off      100.0
dtype: float64

## 5. Visualisierung

### 5a. 2D-Zeitreihe mit farbigen Segmenten und Markern (ein einzelner Beat)

In [11]:
BEAT_ID = int(df_annotations["beat_id"].iloc[20])  # ein Beat etwas nach dem Anfang (Einschwingphase der Filter meiden)

df_markers, df_segments = build_plot_data(df_annotations, beat_id=BEAT_ID, mode="full")

beat_row = df_annotations[df_annotations["beat_id"] == BEAT_ID].iloc[0]
t_center = beat_row["t_R_peak+"]

fig = plot_signal_segments_and_markers(
    df_analysis, signal_col="A_abs",
    df_segments=df_segments, df_markers=df_markers,
    t_start=t_center - 0.4, t_end=t_center + 0.6,
    title=f"Beat {BEAT_ID} — A_abs mit Segmenten und Markern",
)
fig.show()

df_markers:    13 Punkte   (full)
df_segments:    4 Segmente (full)


### 5b. 3D-VCG-Trajektorie desselben Beats

Die farbigen Segmente (P/QRS/T) sollten sich als klar getrennte Schleifen im 3D-Raum abzeichnen — das ist die zentrale Idee hinter dem Frenet-Serret-Annotationsansatz.

In [12]:
fig = plot_vcg_3d(
    df_analysis, df_segments=df_segments, df_markers=df_markers,
    t_start=t_center - 0.4, t_end=t_center + 0.6,
    title=f"Beat {BEAT_ID} — VCG 3D-Trajektorie",
)
fig.show()

---

**Weiter geht's in [`02_feature_extraction.ipynb`](02_feature_extraction.ipynb):** aus `df_analysis` + `df_annotations` werden Beat-zu-Beat-Rotation, P-/QRS-/T-Loop-Features und HRV-Kennwerte berechnet.